# 1. Policy gradient Methods


Policy-gradient methods are RL methods that directly update the policy parameters $\theta$ to maximize expected reward.

The objective is:

$$J(\theta) = \mathbb{E}_{a \sim \pi_\theta}[R(a)]$$

## Notation

| Symbol | Meaning |
|---|---|
| $\theta$ | Trainable parameters of the model (weights) |
| $\pi_\theta$ | The policy — the model's probability distribution over actions, parameterized by $\theta$ |
| $a$ | Action — the discrete choice the model produces (e.g. the token `"Paris"`) |
| $a \sim \pi_\theta$ | "Action $a$ is sampled from the policy $\pi_\theta$" |
| $R(a)$ | Reward received for taking action $a$ |
| $J(\theta)$ | Expected reward under the current policy — the objective we want to **maximize** |
| $\mathbb{E}_{a \sim \pi_\theta}[\cdot]$ | Expectation (average) over actions drawn from the current policy |

## Meaning

Sample actions from the current policy, observe reward, and update the policy so high-reward actions become more likely.

# 2. What is the main challenge?

The model samples a token / action. **Sampling is discrete.**

**Example:**

```
Model samples: "Paris"
Reward: +1
```

The problem is:

How do we backpropagate through a sampled discrete token?

We cannot directly differentiate through the sampling operation. Policy-gradient methods solve this using the **log-probability trick**.

# 3. The log-probability trick

The policy-gradient identity is:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[R(a) \cdot \nabla_\theta \log \pi_\theta(a \mid s)\right]$$

## Notation

| Symbol | Meaning |
|---|---|
| $\theta$ | Trainable parameters of the model (weights) |
| $\pi_\theta$ | The policy — the model's probability distribution over actions, parameterized by $\theta$ |
| $s$ | State — the context the model conditions on (e.g. the prompt) |
| $a$ | Action — the discrete choice the model sampled (e.g. the token `"Paris"`) |
| $\pi_\theta(a \mid s)$ | Probability the policy assigns to action $a$ given state $s$ |
| $\log \pi_\theta(a \mid s)$ | Log-probability of that action — differentiable in $\theta$ |
| $R(a)$ | Reward received for taking action $a$ |
| $J(\theta)$ | Expected reward under the current policy (the objective we want to maximize) |
| $\nabla_\theta$ | Gradient with respect to model parameters $\theta$ |
| $\mathbb{E}[\cdot]$ | Expectation — average over many samples drawn from the policy |

## The key idea

> Instead of differentiating through the sampled action itself, we differentiate the **log-probability** that the model assigned to the sampled action.

Sampling `"Paris"` is a discrete, non-differentiable operation. But the *probability* the model assigned to `"Paris"` — i.e. $\pi_\theta(\text{Paris} \mid s)$ — is a smooth, differentiable function of $\theta$. So we move the gradient off the sample and onto the log-probability.

## What this looks like in practice

If `"Paris"` got **good reward**, we **increase**:

$$\log \pi_\theta(\text{Paris} \mid s)$$

If `"London"` got **bad reward**, we **decrease**:

$$\log \pi_\theta(\text{London} \mid s)$$

The reward $R(a)$ acts as a *scaling factor* on the gradient of the log-probability:

- Positive reward → push log-prob of that action **up** → action becomes more likely next time.
- Negative reward → push log-prob of that action **down** → action becomes less likely next time.


# 4. What is REINFORCE?

REINFORCE is the **simplest policy-gradient algorithm**. It uses the policy-gradient identity directly.

The basic REINFORCE update is:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[R \cdot \nabla_\theta \log \pi_\theta(a \mid s)\right]$$

In **loss form**, because PyTorch minimizes losses:

$$L(\theta) = -R \cdot \log \pi_\theta(a \mid s)$$

## Using advantage instead of raw reward

Usually we use **advantage** instead of raw reward:

$$A = R - b$$

where $b$ is a **baseline**.  You sample N trajectories, compute their rewards, take the mean, and use that as the baseline for all of them.

$$b = \frac{1}{N} \sum_{i=1}^{N} R_i$$

Example: You sampled 4 completions with rewards [10,8,6,4]. Then b=7 and advantages are [+3,+1,−1,−3] 


Then:

$$L(\theta) = -A \cdot \log \pi_\theta(a \mid s)$$

This is the core REINFORCE loss.

## Notation

| Symbol | Meaning |
|---|---|
| $\theta$ | Trainable model parameters |
| $\pi_\theta(a \mid s)$ | Probability the policy assigns to action $a$ in state $s$ |
| $R$ | Reward received for taking action $a$ |
| $b$ | Baseline — a reference reward level we subtract off |
| $A = R - b$ | Advantage — how much *better than the baseline* this action was |
| $L(\theta)$ | The loss we backpropagate through |

# Multiple-token LLM example

**Prompt:**

```
"The capital of France is"
```

**Generated completion:**

```
["Paris", ".", "<eos>"]
```

The probability of the full completion is:

$$\pi_\theta(y \mid x) = \pi_\theta(\text{Paris} \mid x) \cdot \pi_\theta(\text{.} \mid x, \text{Paris}) \cdot \pi_\theta(\text{<eos>} \mid x, \text{Paris .})$$

The log-probability is:

$$\log \pi_\theta(y \mid x) = \log \pi_\theta(\text{Paris} \mid x) + \log \pi_\theta(\text{.} \mid x, \text{Paris}) + \log \pi_\theta(\text{<eos>} \mid x, \text{Paris .})$$

If the completion gets advantage:

$$A = +0.8$$

REINFORCE loss is:

$$L = -A \sum_t \log \pi_\theta(y_t \mid x, y_{<t})$$

For this example:

$$L = -0.8 \left[\log \pi_\theta(\text{Paris}) + \log \pi_\theta(\text{.}) + \log \pi_\theta(\text{<eos>})\right]$$

This increases the probability of the full sampled completion.

In [1]:
import torch
import torch.nn.functional as F

# logits from the current trainable model
# shape: [batch, vocab_size]
logits = torch.tensor([
    [2.0, 0.5, 0.2, 0.1],   # example 1
    [0.1, 2.0, 0.3, 0.2],   # example 2
], requires_grad=True)

# sampled actions from the model
actions = torch.tensor([0, 1])

# advantages from reward/baseline
advantages = torch.tensor([0.8, -0.6])

log_probs = F.log_softmax(logits, dim=-1)

selected_log_probs = log_probs.gather(
    dim=-1,
    index=actions.unsqueeze(-1)
).squeeze(-1)

loss = -(advantages * selected_log_probs).mean()

loss.backward()

print("selected_log_probs:", selected_log_probs)
print("loss:", loss)
print("gradients on logits:", logits.grad)

selected_log_probs: tensor([-0.4305, -0.4038], grad_fn=<SqueezeBackward1>)
loss: tensor(0.0510, grad_fn=<NegBackward0>)
gradients on logits: tensor([[-0.1399,  0.0580,  0.0430,  0.0389],
        [-0.0300,  0.0997, -0.0366, -0.0331]])
